# Chapter 11: Novelty and Outlier Detection
## 📌 Summary
Outlier/anomaly detection mengidentifikasi data points yang berbeda signifikan dari distribusi normal. Penting untuk fraud detection, quality control, dan network security.

## 🧠 Theoretical Deep-Dive

### 11.1 Outlier vs Novelty Detection
- **Outlier detection** (unsupervised): training data mengandung outlier; model mendeteksi outlier dalam data yang sama
- **Novelty detection** (semi-supervised): training data bersih; model mendeteksi data baru yang berbeda

### 11.2 Isolation Forest
Membangun random trees yang **mengisolasi** points. Outlier butuh lebih sedikit splits untuk diisolasi karena terletak di region sparse.
- `contamination`: persentase outlier yang diharapkan
- Score anomaly: rata-rata kedalaman isolasi

### 11.3 One-Class SVM
Belajar **decision boundary** di sekeliling normal data. Points di luar boundary = anomaly.
- `nu`: upper bound pada outlier fraction
- `kernel='rbf'` untuk non-linear boundary

### 11.4 Local Outlier Factor (LOF)
Membandingkan **local density** suatu point dengan tetangganya. Point dengan density jauh lebih rendah dari tetangga = outlier.
- LOF ≈ 1: normal; LOF >> 1: outlier

### 11.5 Elliptic Envelope
Asumsi data mengikuti **Gaussian distribution**. Fit multivariate Gaussian dan flag points dengan Mahalanobis distance tinggi.

## 💻 Code Reproduction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X_normal = np.random.randn(200, 2)
X_outliers = np.random.uniform(low=-6, high=6, size=(20, 2))
X = np.vstack([X_normal, X_outliers])

iso = IsolationForest(n_estimators=100, contamination=0.09, random_state=42)
labels = iso.fit_predict(X)
scores = iso.decision_function(X)

print("Anomaly scores range:", scores.min().round(3), "to", scores.max().round(3))
print("Predicted outliers:", (labels == -1).sum())

plt.figure(figsize=(8, 5))
plt.scatter(X[labels==1, 0], X[labels==1, 1], c="blue", label="Normal", alpha=0.7)
plt.scatter(X[labels==-1, 0], X[labels==-1, 1], c="red", marker="x", s=100, label="Anomaly", linewidths=2)
plt.title("Isolation Forest: Anomaly Detection")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X_normal = np.random.randn(150, 2)
X_outliers = np.random.uniform(-5, 5, (20, 2))
X = np.vstack([X_normal, X_outliers])

scaler = StandardScaler()
X_s = scaler.fit_transform(X)

ocsvm = OneClassSVM(kernel="rbf", nu=0.1, gamma="scale")
ocsvm.fit(X_s[:150])  # train on normal data only

labels = ocsvm.predict(X_s)
print("Predicted outliers:", (labels == -1).sum())

plt.figure(figsize=(8, 5))
plt.scatter(X_s[labels==1, 0], X_s[labels==1, 1], c="blue", label="Normal")
plt.scatter(X_s[labels==-1, 0], X_s[labels==-1, 1], c="red", marker="x", s=100, label="Outlier", linewidths=2)
plt.title("One-Class SVM: Novelty Detection")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
from sklearn.neighbors import LocalOutlierFactor
import matplotlib.pyplot as plt

np.random.seed(42)
X_normal = np.random.randn(150, 2)
X_outliers = np.array([[5, 5], [-5, 5], [5, -5], [-5, -5], [0, 6]])
X = np.vstack([X_normal, X_outliers])

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.03)
labels = lof.fit_predict(X)
lof_scores = -lof.negative_outlier_factor_

print("Predicted outliers:", (labels == -1).sum())
print("LOF scores for last 5 (outliers):", lof_scores[-5:].round(2))
print("LOF scores range (normal):", lof_scores[:150].min().round(2), "to", lof_scores[:150].max().round(2))

plt.figure(figsize=(8, 5))
plt.scatter(X[labels==1, 0], X[labels==1, 1], c="blue", label="Normal", alpha=0.7)
plt.scatter(X[labels==-1, 0], X[labels==-1, 1], c="red", marker="x", s=200, label="Outlier", linewidths=2)
plt.title("Local Outlier Factor")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
from sklearn.covariance import EllipticEnvelope
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import f1_score

np.random.seed(42)
X_normal = np.random.randn(200, 2)
X_outliers = np.random.uniform(-5, 5, (20, 2))
X = np.vstack([X_normal, X_outliers])
y_true = np.array([1]*200 + [-1]*20)

models = [
    ("IsolationForest", IsolationForest(contamination=0.09, random_state=42)),
    ("OneClassSVM", OneClassSVM(nu=0.1, gamma="scale")),
    ("LOF", LocalOutlierFactor(contamination=0.09)),
    ("EllipticEnvelope", EllipticEnvelope(contamination=0.09, random_state=42))
]

for name, model in models:
    if name == "LOF":
        y_pred = model.fit_predict(X)
    else:
        model.fit(X[:200])
        y_pred = model.predict(X)
    f1 = f1_score(y_true, y_pred, pos_label=-1)
    n_detected = (y_pred == -1).sum()
    print(f"{name:20}: detected={n_detected:2}, F1(anomaly)={f1:.4f}")

## ✅ Chapter Summary
- **Isolation Forest**: best general-purpose, scalable, minimal hyperparameters
- **One-Class SVM**: powerful untuk novelty detection, slow untuk large data
- **LOF**: best untuk local density anomalies, interpretable
- **EllipticEnvelope**: hanya jika data Gaussian
- Set `contamination` sesuai domain knowledge
- Evaluasi: sulit tanpa label; gunakan F1-score jika ada ground truth
- Kombinasikan beberapa detector (ensemble) untuk robustness